In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset, RandomSampler
from sklearn.model_selection import train_test_split

# Cek device
device = torch.device('cpu')
print(f"Perangkat: {device}")

# 1. Load Tokenizer DistilBERT (Versi Multilingual)
print("Sedang mendownload Tokenizer DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')

# 2. Load Model DistilBERT
print("Sedang mendownload Model DistilBERT...")
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels=2
)
model.to(device)

print("✅ DistilBERT Siap Digunakan!")

d:\SMESTER7\Machine Learning C\UAP\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Perangkat: cpu
Sedang mendownload Tokenizer DistilBERT...


d:\SMESTER7\Machine Learning C\UAP\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Sedang mendownload Model DistilBERT...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ DistilBERT Siap Digunakan!


In [2]:
# Load Data
df = pd.read_csv('../DATASETUAP/data_bersih.csv')
df.dropna(inplace=True)

sentences = df['text_clean'].values
labels = df['label'].values

# Fungsi Tokenisasi Khusus DistilBERT
def convert_data_to_bert_input(sentences, labels):
    input_ids = []
    attention_masks = []

    for sent in sentences:
        encoded_dict = tokenizer.encode_plus(
            str(sent),
            add_special_tokens=True,
            max_length=64, 
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)

    return input_ids, attention_masks, labels

print("Memproses data... (Tunggu sebentar)")
input_ids, attention_masks, labels = convert_data_to_bert_input(sentences, labels)

# Split & DataLoader
dataset = TensorDataset(input_ids, attention_masks, labels)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

batch_size = 16
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)

print("✅ Data Siap Training!")

Memproses data... (Tunggu sebentar)
✅ Data Siap Training!


In [3]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 2 # Cukup 2 epoch

print("🚀 MULAI TRAINING DISTILBERT...")
print("Estimasi: Sedikit lebih cepat dari IndoBERT tadi.")

for epoch_i in range(0, epochs):
    print(f'\n======== Epoch {epoch_i + 1} / {epochs} ========')
    print('Training...')
    total_train_loss = 0
    model.train()
    
    for step, batch in enumerate(train_dataloader):
        if step % 50 == 0 and not step == 0:
            print(f'  Batch {step} dari {len(train_dataloader)}...')

        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        model.zero_grad()
        # DistilBERT tidak butuh token_type_ids
        result = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
        
        loss = result.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"  Rata-rata Loss: {avg_train_loss:.2f}")

print("\n✅ Training DistilBERT Selesai!")

🚀 MULAI TRAINING DISTILBERT...
Estimasi: Sedikit lebih cepat dari IndoBERT tadi.

======== Epoch 1 / 2 ========
Training...
  Batch 50 dari 659...
  Batch 100 dari 659...
  Batch 150 dari 659...
  Batch 200 dari 659...
  Batch 250 dari 659...
  Batch 300 dari 659...
  Batch 350 dari 659...
  Batch 400 dari 659...
  Batch 450 dari 659...
  Batch 500 dari 659...
  Batch 550 dari 659...
  Batch 600 dari 659...
  Batch 650 dari 659...
  Rata-rata Loss: 0.47

======== Epoch 2 / 2 ========
Training...
  Batch 50 dari 659...
  Batch 100 dari 659...
  Batch 150 dari 659...
  Batch 200 dari 659...
  Batch 250 dari 659...
  Batch 300 dari 659...
  Batch 350 dari 659...
  Batch 400 dari 659...
  Batch 450 dari 659...
  Batch 500 dari 659...
  Batch 550 dari 659...
  Batch 600 dari 659...
  Batch 650 dari 659...
  Rata-rata Loss: 0.27

✅ Training DistilBERT Selesai!


In [4]:
import os

output_dir = '../models/model_distilbert/'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Simpan
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ SUKSES! Model DistilBERT tersimpan.")
print("Sekarang kamu punya 3 Model LENGKAP sesuai syarat modul!")

✅ SUKSES! Model DistilBERT tersimpan.
Sekarang kamu punya 3 Model LENGKAP sesuai syarat modul!
